<div style='background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%); padding: 44px 48px 38px; border-radius: 14px; margin-bottom: 4px;'>
  <h1 style='color: #ffffff; margin: 0 0 10px; font-size: 2.3em; letter-spacing: -0.5px; font-weight: 700;'>
    MYH Yrkesh&ouml;gskolan , Application Data Pipeline
  </h1>
  <p style='color: #b0c8de; margin: 0; font-size: 1.05em; line-height: 1.9;'>
    A curated, harmonized dataset built from eight years of higher vocational education application results.<br>
    <strong style='color: #7ec8e3;'>2018 &ndash; 2025 &nbsp;&middot;&nbsp; 8 years &nbsp;&middot;&nbsp; 9&thinsp;983 records &nbsp;&middot;&nbsp; MYH Data Pipeline Project</strong>
  </p>
</div>

## Overview

**[Myndigheten för yrkeshögskolan (MYH)](https://www.myh.se/yrkeshogskolan/resultat-ansokningsomgangar/resultat-for-program)** publishes one Excel file per application round, containing decisions on which higher vocational education programs were approved or rejected. The files span many years but are not consistent , column names, sheet layouts, and available fields changed over time.

| | |
|:---|:---|
| **Data source** | MYH public Excel files , application round results |
| **Scope** | 2018 – 2025 (8 application years) |
| **Main sheet** | `Tabell 3` (2020–2025) · `Lista` (2018–2019) |
| **Output** | `data/curated_applications.csv` |
| **Database** | PostgreSQL , `data_pipeline_de25` |

### What this notebook does

1. Load raw files and document structural differences across years
2. Define a universal harmonization schema
3. Apply column renaming, value normalization, and type conversion
4. Clean and standardize the unified dataset
5. Enrich with computed columns
6. Validate quality with automated checks
7. Export the final curated dataset
8. Pipeline Report
9. Reflection

In [1]:
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Suppress openpyxl "no default style" warnings - cosmetic, no effect on data.
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = Path.cwd()
RAW_PATH = BASE / "resultat_ansokning"
OUT_PATH = BASE / "data"
OUT_PATH.mkdir(exist_ok=True)

# ── Per-year source configuration ─────────────────────────────────────────────
# MYH changed file structure across years:
#   2018-2019 : sheet "Lista",    header row 0
#   2020-2022 : sheet "Tabell 3", header row 0
#   2023-2024 : sheet "Tabell 3", header row 5  (merged title rows above data)
#   2025      : sheet "Tabell 3", header row 6  (one extra merged row added)
YEAR_CONFIG = {
    2018: {
        "file": "beslut-for-samtliga-ansokningar-sorterade-efter-utbildningsomrade-och-utbildningsnamn-2018.xlsx",
        "sheet": "Lista",
        "header": 0,
    },
    2019: {
        "file": "beslut-for-samtliga-ansokningar-sorterade-efter-utbildningsomrade-och-utbildningsnamn-2019.xlsx",
        "sheet": "Lista",
        "header": 0,
    },
    2020: {
        "file": "resultat-ansokningsomgang-2020.xlsx",
        "sheet": "Tabell 3",
        "header": 0,
    },
    2021: {
        "file": "resultat-ansokningsomgang-2021.xlsx",
        "sheet": "Tabell 3",
        "header": 0,
    },
    2022: {
        "file": "resultat-ansokningsomgang-2022.xlsx",
        "sheet": "Tabell 3",
        "header": 0,
    },
    2023: {
        "file": "resultat-ansokningsomgang-2023.xlsx",
        "sheet": "Tabell 3",
        "header": 5,
    },
    2024: {
        "file": "resultat-ansokningsomgang-2024.xlsx",
        "sheet": "Tabell 3",
        "header": 5,
    },
    2025: {
        "file": "resultat-ansokningsomgang-2025.xlsx",
        "sheet": "Tabell 3",
        "header": 6,
    },
}

print(
    f"Configuration loaded - {len(YEAR_CONFIG)} years ({min(YEAR_CONFIG)}-{max(YEAR_CONFIG)})"
)
print(f"Source : {RAW_PATH}")
print(f"Output : {OUT_PATH}")

Configuration loaded - 8 years (2018-2025)
Source : myh-data-pipeline\resultat_ansokning
Output : myh-data-pipeline\data


In [2]:
# ── Pipeline event log ────────────────────────────────────────────────────────
# Shared list populated by every step. Use log_event() to append from any cell.
# The final section (Pipeline Report) renders the full log as a colour-coded table.
PIPELINE_LOG: list[dict] = []


def log_event(step: str, level: str, message: str, detail: str = "") -> None:
    """Append one event to the pipeline log.

    level: "ok" | "warning" | "error"
    """
    PIPELINE_LOG.append(
        {"Step": step, "Level": level, "Message": message, "Detail": detail}
    )


# ── Setup checks ──────────────────────────────────────────────────────────────
if not RAW_PATH.exists():
    log_event("setup", "error", "Source directory not found", str(RAW_PATH))
else:
    missing_files = [
        cfg["file"]
        for cfg in YEAR_CONFIG.values()
        if not (RAW_PATH / cfg["file"]).exists()
    ]
    if missing_files:
        for fname in missing_files:
            log_event("setup", "error", "Source file missing", fname)
    else:
        log_event("setup", "ok", f"All {len(YEAR_CONFIG)} source files found")

setup_errors = sum(1 for e in PIPELINE_LOG if e["Level"] == "error")
print(
    f"Setup: {'OK' if not setup_errors else f'{setup_errors} error(s) - see Pipeline Report for details'}"
)

Setup: OK


---
## 1. Data Source & Raw Exploration

### Why 2018 – 2025?

Files published before 2018 are missing critical columns , `Studieform`, `Studietakt %`, and `Huvudmannatyp` are either absent or inconsistently structured. Including them would introduce large null gaps in the most analytically useful fields. 2018 is the earliest year where all core columns are consistently present.

Going back to 2018 also adds two years that use a different sheet layout (`Lista` instead of `Tabell 3`), which makes for a richer harmonization story and a more robust pipeline.

### Why Tabell 3 / Lista?

`Tabell 3` is the application-level table , one row per application, which is exactly what we want. `Tabell 4` exists in later years but represents municipality-level detail where a single application can appear multiple times. Mixing them would corrupt aggregations.

For 2018–2019, the equivalent table is named `Lista` , it carries the same application-level granularity.

In [3]:
raw_dataframes: dict[int, pd.DataFrame] = {}
raw_column_names: dict[int, list[str]] = {}
exploration_rows = []

for year, config in YEAR_CONFIG.items():
    try:
        year_df = pd.read_excel(
            RAW_PATH / config["file"],
            sheet_name=config["sheet"],
            header=config["header"],
        )
    except FileNotFoundError:
        log_event("explore", "error", f"{year}: file not found", config["file"])
        continue
    except Exception as exc:
        log_event("explore", "error", f"{year}: could not read file", str(exc))
        continue

    # Drop empty columns and embedded notes/junk columns (common in 2019)
    year_df = year_df.dropna(axis=1, how="all")
    year_df = year_df[
        [
            c
            for c in year_df.columns
            if not str(c).startswith("Unnamed") and not str(c).startswith("Notera")
        ]
    ]

    if len(year_df) == 0:
        log_event("explore", "error", f"{year}: 0 rows after loading")
        continue
    if len(year_df) < 100:
        log_event(
            "explore",
            "warning",
            f"{year}: unexpectedly few rows",
            f"{len(year_df)} rows",
        )

    if "Beslut" not in year_df.columns:
        log_event(
            "explore", "warning", f"{year}: 'Beslut' column missing from raw file"
        )

    raw_dataframes[year] = year_df
    raw_column_names[year] = list(year_df.columns)

    beslut_counts = (
        year_df["Beslut"].value_counts() if "Beslut" in year_df.columns else {}
    )
    approved_count = sum(
        v
        for k, v in beslut_counts.items()
        if "Beviljad" in str(k) and "Ej" not in str(k)
    )
    total_rows = len(year_df)

    exploration_rows.append(
        {
            "Year": year,
            "Sheet": config["sheet"],
            "Header row": config["header"],
            "Rows": total_rows,
            "Columns": len(year_df.columns),
            "Approved": approved_count,
            "Approval %": f"{approved_count / total_rows:.0%}" if total_rows else "-",
        }
    )

file_summary = pd.DataFrame(exploration_rows).set_index("Year")
print(
    f"Loaded {len(raw_dataframes)} / {len(YEAR_CONFIG)} files, {file_summary['Rows'].sum():,} total rows\n"
)
display(
    file_summary.style.background_gradient(subset=["Rows", "Approved"], cmap="Blues")
)

Loaded 8 / 8 files, 9,983 total rows



,Sheet,Header row,Rows,Columns,Approved,Approval %
Year,,,,,,
2018,Lista,0,1105,11,496,45%
2019,Lista,0,1237,13,482,39%
2020,Tabell 3,0,1482,16,484,33%
2021,Tabell 3,0,1238,17,426,34%
2022,Tabell 3,0,1207,16,420,35%
2023,Tabell 3,5,1258,28,477,38%
2024,Tabell 3,5,1272,28,344,27%
2025,Tabell 3,6,1184,28,462,39%


In [4]:
# Map each logical field to a lambda that checks whether it exists in a year's
# column list. Different heading variants across years are handled here - this
# matrix is the evidence base for every mapping decision in COLUMN_MAP.
PRESENCE_CHECKS = {
    "Utbildningsnamn": lambda column_names: "Utbildningsnamn" in column_names,
    "Utbildningsomr.": lambda column_names: any(
        "Utbildningsomr" in col for col in column_names
    ),
    "Beslut": lambda column_names: "Beslut" in column_names,
    "Diarienummer": lambda column_names: "Diarienummer" in column_names,
    "Län": lambda column_names: "Län" in column_names or "Lan" in column_names,
    "Kommun": lambda column_names: "Kommun" in column_names,
    "Studieform": lambda column_names: "Studieform" in column_names,
    "Studietakt %": lambda column_names: "Studietakt %" in column_names,
    "YH-poäng": lambda column_names: any("YH-po" in col for col in column_names),
    "Typ av examen": lambda column_names: any(
        heading in column_names for heading in ["Typ av examen", "Examenstyp"]
    ),
    "Anordnare": lambda column_names: any("nordnare" in col for col in column_names),
    "Huvudmannatyp": lambda column_names: "Huvudmannatyp" in column_names,
    "Flera kommuner": lambda column_names: any(
        heading in column_names
        for heading in ["Flera kommuner", "Flera studiekommuner", "Flera studieorter"]
    ),
    "SUN5 inriktning": lambda column_names: any("SUN5" in col for col in column_names),
    "SeQF nivå": lambda column_names: any("SeQF" in col for col in column_names),
}

presence_matrix = pd.DataFrame(
    {
        year: {
            field: check(raw_column_names[year])
            for field, check in PRESENCE_CHECKS.items()
        }
        for year in YEAR_CONFIG
    }
).T
presence_matrix.index.name = "Year"


def _style_presence(is_present: bool) -> str:
    """Green = column present this year; red = absent."""
    if is_present:
        return "background-color: rgba(46, 204, 113, 0.30); font-weight: 600; text-align: center"
    return "background-color: rgba(231, 76, 60, 0.25); text-align: center"


display(
    presence_matrix.style.map(_style_presence).format(
        lambda is_present: "Yes" if is_present else "No"
    )
)

,Utbildningsnamn,Utbildningsomr.,Beslut,Diarienummer,Län,Kommun,Studieform,Studietakt %,YH-poäng,Typ av examen,Anordnare,Huvudmannatyp,Flera kommuner,SUN5 inriktning,SeQF nivå
Year,,,,,,,,,,,,,,,
2018,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,No,No,No
2019,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,No,No
2020,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,No,No
2021,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No,No
2022,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No,No
2023,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
2024,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
2025,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes


---
## 2. Harmonization Design

The core challenge is that eight years of files were never designed to be combined. Column names changed, value vocabularies drifted, and new fields were introduced. Below is the complete mapping from source names to our canonical schema.

### Column Name Mapping

| Canonical column | Source variants across years |
|:---|:---|
| `utbildningsomrade` | `Utbildningsområde` (all years) |
| `utbildningsnamn` | `Utbildningsnamn` (all years) |
| `beslut` | `Beslut` (all years) |
| `diarienummer` | `Diarienummer` (all years) |
| `lan` | `Län` (all years) |
| `kommun` | `Kommun` (all years) |
| `yh_poang` | `YH-poäng` (all years) |
| `studieform` | `Studieform` (2018–2025) |
| `studietakt_procent` | `Studietakt %` (2018–2025) |
| `typ_av_examen` | `Typ av examen` (2021–2024) · `Examenstyp` (2025) · *null* (2018–2020) |
| `utbildningsanordnare` | `Anordnare administrativ enhet` (2018) · `Utbildningsanordnare, administrativ enhet` (2019) · `Utbildningsanordnare administrativ enhet` (2020–2025) |
| `huvudmannatyp` | `Huvudmannatyp` (all years) |
| `flera_kommuner` | `Flera kommuner` (2022–2025) · `Flera studiekommuner` (2020–2021) · `Flera studieorter` (2019) · *null* (2018) |
| `antal_kommuner` | `Antal kommuner` (2020–2025) · `Antal studieorter` (2019) · *null* (2018) |
| `sokta_omgangar` | `Sökta utbildningsomgångar` (2020–2025) · *null* (2018–2019) |
| `beviljade_omgangar` | `Beviljade utbildningsomgångar` (2020–2025) · *null* (2018–2019) |
| `sun5_inriktning` | `SUN5 inriktning` (2023–2025) · *null* (2018–2022) |
| `sun5_inriktning_namn` | `SUN5 inriktning namn` (2023–2025) · *null* (2018–2022) |
| `seqf_niva` | `SeQF nivå` (2023–2025) · *null* (2018–2022) |
| `smalt_yrkesomrade` | `Smalt yrkesområde` (2023–2025) · *null* (2018–2022) |

### Value Normalization

| Field | Raw values | Normalized to |
|:---|:---|:---|
| `beslut` -> `beslut_normalized` | `Beviljad` | `approved` |
| | `Avslag` (2022–2025) | `rejected` |
| | `Ej beviljad` (2018–2021) | `rejected` |
| | `Återkallad` (2025, 1 row) | `withdrawn` |
| `huvudmannatyp` | `Landsting` (2018–2021) | `Region` |

<div style='border-left: 4px solid #d4a017; padding: 12px 16px; border-radius: 4px; margin: 16px 0; background: rgba(212, 160, 23, 0.1);'>
<strong>Decision note:</strong> Columns that only exist in later years (SUN5, SeQF) are retained and will be <code>null</code> for earlier years. A null clearly signals absence of data; dropping these columns would silently hide the fact that they exist at all. They are important but not vital.
</div>

<div style='border-left: 4px solid #8e6bbf; padding: 12px 16px; border-radius: 4px; margin: 16px 0; background: rgba(142, 107, 191, 0.1);'>
<strong>Återkallad:</strong> One application in 2025 has the decision value <code>Återkallad</code> (Swedish: "withdrawn"). This means the provider withdrew the application before a decision was issued. It is distinct from a rejection and is mapped to <code>withdrawn</code> rather than <code>rejected</code> to preserve the semantic difference.
</div>

In [5]:
# ── Harmonization constants ───────────────────────────────────────────────────
# MYH changed column names across years without a published mapping.
# COLUMN_MAP captures every name variant found in the raw files and maps them all
# to the same canonical snake_case key, so every year ends up with identical columns.
COLUMN_MAP = {
    "Utbildningsområde": "utbildningsomrade",
    "Utbildningsnamn": "utbildningsnamn",
    "Beslut": "beslut",
    "Diarienummer": "diarienummer",
    "Län": "lan",
    "Kommun": "kommun",
    "YH-poäng": "yh_poang",
    "Studieform": "studieform",
    "Studietakt %": "studietakt_procent",
    # "Typ av examen" in 2020+, "Examenstyp" in earlier files
    "Typ av examen": "typ_av_examen",
    "Examenstyp": "typ_av_examen",
    # Provider column appeared under three different headings across the years
    "Anordnare administrativ enhet": "utbildningsanordnare",
    "Utbildningsanordnare, administrativ enhet": "utbildningsanordnare",
    "Utbildningsanordnare administrativ enhet": "utbildningsanordnare",
    "Huvudmannatyp": "huvudmannatyp",
    # Multi-municipality flag: three heading variants across 2019-2025
    "Flera kommuner": "flera_kommuner",
    "Flera studiekommuner": "flera_kommuner",
    "Flera studieorter": "flera_kommuner",
    # Municipality count: two heading variants
    "Antal kommuner": "antal_kommuner",
    "Antal studieorter": "antal_kommuner",
    "Sökta utbildningsomgångar": "sokta_omgangar",
    "Beviljade utbildningsomgångar": "beviljade_omgangar",
    # SUN5 / SeQF introduced in 2023 , absent in earlier files
    "SUN5 inriktning": "sun5_inriktning",
    "SUN5 inriktning namn": "sun5_inriktning_namn",
    "SeQF nivå": "seqf_niva",
    "Smalt yrkesområde": "smalt_yrkesomrade",
}

# Maps raw Swedish decision values to the three canonical English labels.
# "Ej beviljad " (trailing space) is a data-quality quirk found in older files.
# "Återkallad" (withdrawn) was observed in 2025 , the provider cancelled the application.
BESLUT_MAP = {
    "Beviljad": "approved",
    "Avslag": "rejected",
    "Ej beviljad": "rejected",
    "Ej beviljad ": "rejected",  # trailing-space variant in older files
    "Återkallad": "withdrawn",  # provider withdrew application (observed in 2025)
}

# "Landsting" was renamed to "Region" across Sweden in 2019.
# The source files mix both terms depending on the year of the application.
HUVUDMAN_MAP = {"Landsting": "Region"}

# Explicit allowlist of columns to keep after harmonization.
# Columns present in some years but absent in others are still included here
# so they appear as NaN for the years where they did not exist, rather than
# being silently dropped.
KEEP_COLUMNS = [
    "source_year",
    "source_file",
    "source_sheet",
    "diarienummer",
    "utbildningsnamn",
    "utbildningsomrade",
    "beslut",
    "lan",
    "kommun",
    "yh_poang",
    "studieform",
    "studietakt_procent",
    "typ_av_examen",
    "utbildningsanordnare",
    "huvudmannatyp",
    "flera_kommuner",
    "antal_kommuner",
    "sokta_omgangar",
    "beviljade_omgangar",
    "sun5_inriktning",
    "sun5_inriktning_namn",
    "seqf_niva",
    "smalt_yrkesomrade",
]

print(f"{len(COLUMN_MAP)} column name mappings defined")
print(f"{len(BESLUT_MAP)} beslut value mappings defined")
print(f"{len(KEEP_COLUMNS)} canonical columns in final schema")

26 column name mappings defined
5 beslut value mappings defined
23 canonical columns in final schema


---
## 3. Load & Harmonize

Each year is loaded using its entry in `YEAR_CONFIG` and passed through `load_and_harmonize` , a single function that handles all structural differences in one place:

1. Reads the correct sheet with the correct header offset
2. Drops empty columns and junk columns (notes embedded in the 2019 file)
3. Renames every recognised column using `COLUMN_MAP`
4. Attaches three source traceability columns: `source_year`, `source_file`, `source_sheet`
5. Returns only the columns defined in `KEEP_COLUMNS`

All eight frames are then concatenated into one DataFrame. Columns absent in a given year become `NaN` automatically , `pd.concat` handles this without any special case code.

In [6]:
def load_and_harmonize(year: int, config: dict) -> pd.DataFrame:
    """Load one year's Excel file and rename columns to the canonical schema.

    Each year is loaded independently so structural differences (sheet name,
    header row, column naming) are fully isolated to the YEAR_CONFIG dict.
    Value cleaning and enrichment happen in later steps.
    """
    year_df = pd.read_excel(
        RAW_PATH / config["file"], sheet_name=config["sheet"], header=config["header"]
    )
    year_df = year_df.dropna(axis=1, how="all")
    year_df = year_df[
        [
            c
            for c in year_df.columns
            if not str(c).startswith("Unnamed") and not str(c).startswith("Notera")
        ]
    ]
    year_df = year_df.rename(columns=COLUMN_MAP)
    year_df["source_year"] = year
    year_df["source_file"] = config["file"]
    year_df["source_sheet"] = config["sheet"]
    columns_to_keep = [col for col in KEEP_COLUMNS if col in year_df.columns]
    return year_df[columns_to_keep]


year_frames = []
load_report_rows = []

for year, config in YEAR_CONFIG.items():
    try:
        frame = load_and_harmonize(year, config)
    except FileNotFoundError:
        log_event("harmonize", "error", f"{year}: file not found", config["file"])
        continue
    except Exception as exc:
        log_event("harmonize", "error", f"{year}: harmonization failed", str(exc))
        continue

    if len(frame) == 0:
        log_event("harmonize", "error", f"{year}: 0 rows after harmonization")
        continue

    # Verify that core columns were successfully mapped by COLUMN_MAP
    expected_core = {"diarienummer", "utbildningsnamn", "beslut", "source_year"}
    missing_core = expected_core - set(frame.columns)
    if missing_core:
        log_event(
            "harmonize",
            "warning",
            f"{year}: core columns not found after rename",
            str(missing_core),
        )
    else:
        log_event(
            "harmonize",
            "ok",
            f"{year}: harmonized",
            f"{len(frame)} rows, {len(frame.columns)} columns",
        )

    # Log any KEEP_COLUMNS absent after renaming.
    # Year-gated columns (sun5, seqf, typ_av_examen, round counts) will appear here
    # for years where MYH had not yet introduced those fields - that is expected.
    # An unexpected column name here signals a silent COLUMN_MAP failure.
    missing_keep = set(KEEP_COLUMNS) - set(frame.columns)
    if missing_keep:
        log_event(
            "harmonize",
            "warning",
            f"{year}: {len(missing_keep)} KEEP_COLUMNS absent after rename",
            ", ".join(sorted(missing_keep)),
        )

    year_frames.append(frame)
    load_report_rows.append(
        {"Year": year, "Rows loaded": len(frame), "Columns": len(frame.columns)}
    )

if not year_frames:
    log_event("harmonize", "error", "No frames loaded - pipeline cannot continue")
    raise RuntimeError("Pipeline aborted: no data loaded. Check setup errors above.")

applications = pd.concat(year_frames, ignore_index=True)

load_summary = pd.DataFrame(load_report_rows).set_index("Year")
load_summary.loc["TOTAL"] = load_summary.sum()
load_summary.loc["TOTAL", "Columns"] = len(applications.columns)

print(f"Combined: {applications.shape[0]:,} rows x {applications.shape[1]} columns\n")
display(load_summary.style.background_gradient(subset=["Rows loaded"], cmap="Blues"))

Combined: 9,983 rows x 23 columns



,Rows loaded,Columns
Year,,
2018,1105,14
2019,1237,16
2020,1482,18
2021,1238,19
2022,1207,19
2023,1258,23
2024,1272,23
2025,1184,23
TOTAL,9983,23


---
## 4. Data Cleaning

Harmonization maps columns to a common schema. Cleaning fixes the *values* inside those columns. We handle four categories:

- **Type conversion** , numeric fields stored as strings or mixed types
- **Text normalization** , leading/trailing whitespace
- **Value standardization** , `Landsting` -> `Region`
- **Duplicate investigation** , check for repeated `diarienummer` values

In [7]:
# ── Type coercion ─────────────────────────────────────────────────────────────
# pd.to_numeric coerces unparseable values to NaN.
# Int64 (capital I) is pandas' nullable integer - preserves NaN without float promotion.
# Compare null counts before/after to detect coercion failures beyond expected structural nulls.
numeric_columns = [
    "yh_poang",
    "studietakt_procent",
    "antal_kommuner",
    "sokta_omgangar",
    "beviljade_omgangar",
]
for col in numeric_columns:
    if col not in applications.columns:
        log_event("clean", "warning", f"{col}: column missing, skipping type coercion")
        continue
    before_nulls = int(applications[col].isnull().sum())
    applications[col] = pd.to_numeric(applications[col], errors="coerce").astype(
        "Int64"
    )
    new_nulls = int(applications[col].isnull().sum()) - before_nulls
    if new_nulls > 0:
        log_event(
            "clean",
            "warning",
            f"{col}: {new_nulls} value(s) could not be parsed as numeric",
        )
    else:
        log_event("clean", "ok", f"{col}: type coercion clean")

# ── Text normalization ────────────────────────────────────────────────────────
# Strip whitespace and convert "nan" string (produced when a blank Excel cell is
# cast to str) back to None so the database receives NULL, not the string "nan".
text_columns = [
    "utbildningsnamn",
    "utbildningsomrade",
    "beslut",
    "diarienummer",
    "lan",
    "kommun",
    "studieform",
    "typ_av_examen",
    "utbildningsanordnare",
    "huvudmannatyp",
    "flera_kommuner",
]
for col in text_columns:
    if col in applications.columns:
        applications[col] = (
            applications[col]
            .astype(str)
            .str.strip()
            .replace({"nan": None, "None": None})
        )

# ── Value standardization ─────────────────────────────────────────────────────
# Sweden renamed "Landsting" to "Region" in 2019 - source files use both terms.
if "huvudmannatyp" in applications.columns:
    landsting_count = int((applications["huvudmannatyp"] == "Landsting").sum())
    applications["huvudmannatyp"] = applications["huvudmannatyp"].replace(HUVUDMAN_MAP)
    if landsting_count > 0:
        log_event("clean", "ok", f"Replaced {landsting_count} 'Landsting' -> 'Region'")
else:
    log_event(
        "clean",
        "warning",
        "huvudmannatyp: column missing, Landsting -> Region not applied",
    )

# ── Null summary ──────────────────────────────────────────────────────────────
# Only show columns with at least one null. High null % for year-gated columns
# (SUN5, SeQF, typ_av_examen, round counts) is expected - not a data error.
null_pct = (applications.isnull().mean() * 100).round(1)
null_summary = pd.DataFrame({"Null %": null_pct, "Count": applications.isnull().sum()})[
    null_pct > 0
].sort_values("Null %", ascending=False)

if "diarienummer" in applications.columns:
    print(
        f"Duplicate diarienummer (expected - 2019 multi-municipality rows): "
        f"{applications.duplicated(subset=['diarienummer']).sum():,}"
    )
    print(f"Unique diarienummer values : {applications['diarienummer'].nunique():,}")
print(f"Rows after cleaning        : {len(applications):,}\n")
display(
    null_summary.style.bar(
        subset=["Null %"], color="rgba(231,76,60,0.3)", vmin=0, vmax=100
    ).format({"Null %": "{:.1f}%", "Count": "{:,}"})
)

Duplicate diarienummer (expected - 2019 multi-municipality rows): 140
Unique diarienummer values : 9,843
Rows after cleaning        : 9,983



,Null %,Count
seqf_niva,63.3%,"6,322"
sun5_inriktning,62.8%,"6,269"
smalt_yrkesomrade,62.8%,"6,269"
sun5_inriktning_namn,62.8%,"6,269"
typ_av_examen,38.3%,"3,824"
beviljade_omgangar,23.5%,"2,342"
sokta_omgangar,23.5%,"2,342"
flera_kommuner,11.1%,"1,105"
antal_kommuner,11.1%,"1,105"


<div style='border-left: 4px solid #5b9bd5; padding: 12px 16px; border-radius: 4px; margin: 8px 0; background: rgba(91, 155, 213, 0.1);'>
<strong>Null interpretation:</strong> High null rates in <code>sun5_inriktning</code>, <code>seqf_niva</code>, and <code>typ_av_examen</code> are expected and correct , these columns did not exist in earlier file versions. Nulls here mean the field was not collected in that year, not that data is missing.
</div>

---
## 5. Enrichment

Enrichment goes beyond cleaning , it derives new information from existing data to make the dataset more useful for consumers. We add three computed columns:

| New column | Type | Description |
|:---|:---|:---|
| `beslut_normalized` | `str` | English decision label: `approved` or `rejected` |
| `is_distance` | `bool` | True if `studieform` contains *Distans* |
| `has_multiple_municipalities` | `bool` | True if `flera_kommuner` is *Ja* |

In [8]:
# ── Normalized decision label ─────────────────────────────────────────────────
# Map raw Swedish beslut values to three canonical English labels.
# Rows whose beslut value is not in BESLUT_MAP become NaN - caught below and logged.
if "beslut" in applications.columns:
    applications["beslut_normalized"] = applications["beslut"].map(BESLUT_MAP)
    null_beslut = int(applications["beslut_normalized"].isnull().sum())
    if null_beslut > 0:
        unknown_values = applications.loc[
            applications["beslut_normalized"].isnull(), "beslut"
        ].value_counts()
        log_event(
            "enrich",
            "warning",
            f"beslut_normalized: {null_beslut} row(s) could not be mapped",
            str(unknown_values.to_dict()),
        )
    else:
        log_event("enrich", "ok", "beslut_normalized: all values mapped")
else:
    applications["beslut_normalized"] = None
    log_event(
        "enrich",
        "warning",
        "beslut_normalized: source column 'beslut' missing, set to null",
    )

# ── Distance flag ─────────────────────────────────────────────────────────────
# True for any studieform containing "Distans" (case-insensitive).
# fillna("") prevents NaN from propagating into the boolean result.
if "studieform" in applications.columns:
    applications["is_distance"] = (
        applications["studieform"].fillna("").str.lower().str.contains("distans")
    )
    log_event(
        "enrich",
        "ok",
        f"is_distance: {int(applications['is_distance'].sum())} distance programs",
    )
else:
    applications["is_distance"] = False
    log_event(
        "enrich",
        "warning",
        "is_distance: source column 'studieform' missing, set to False",
    )

# ── Multiple municipalities flag ──────────────────────────────────────────────
# True when flera_kommuner is "Ja" - both "Ja" and "ja" appear in source data.
if "flera_kommuner" in applications.columns:
    applications["has_multiple_municipalities"] = (
        applications["flera_kommuner"].fillna("").str.strip().str.lower().eq("ja")
    )
    log_event(
        "enrich",
        "ok",
        f"has_multiple_municipalities: {int(applications['has_multiple_municipalities'].sum())} programs",
    )
else:
    applications["has_multiple_municipalities"] = False
    log_event(
        "enrich",
        "warning",
        "has_multiple_municipalities: source column 'flera_kommuner' missing, set to False",
    )

print(
    f"Schema after enrichment: {len(applications.columns)} columns  "
    f"({len(KEEP_COLUMNS)} base + 3 derived)\n"
)

enrich_summary = pd.DataFrame(
    [
        {
            "Column": col,
            "Values": str(applications[col].value_counts().to_dict()),
        }
        for col in ("beslut_normalized", "is_distance", "has_multiple_municipalities")
    ]
).set_index("Column")

display(enrich_summary)

Schema after enrichment: 26 columns  (23 base + 3 derived)



,Values
Column,
beslut_normalized,"{'rejected': 6391, 'approved': 3591, 'withdrawn': 1}"
is_distance,"{False: 5722, True: 4261}"
has_multiple_municipalities,"{False: 8324, True: 1659}"


---
## 6. Validation

Before export, eight automated quality checks verify the dataset meets expected thresholds. Each check is recorded with a pass/fail status and a detail message , a green table here means the pipeline is working correctly end-to-end.

A check that fails signals a problem in the pipeline, not something to work around.

In [9]:
quality_checks: list[dict] = []


def record(name: str, passed: bool, detail: str = "") -> None:
    """Record one quality check result."""
    quality_checks.append(
        {"Check": name, "Status": "Pass" if passed else "Fail", "Detail": detail}
    )


def has_column(col: str) -> bool:
    return col in applications.columns


record(
    "All years loaded",
    applications["source_year"].nunique() == len(YEAR_CONFIG),
    f"{applications['source_year'].nunique()} / {len(YEAR_CONFIG)} years present",
)
record(
    "No completely empty rows",
    not applications.isnull().all(axis=1).any(),
    f"{applications.isnull().all(axis=1).sum()} fully-null rows",
)

if has_column("beslut_normalized"):
    record(
        "beslut_normalized fully populated",
        applications["beslut_normalized"].isnull().sum() == 0,
        f"{applications['beslut_normalized'].isnull().sum()} nulls",
    )
    record(
        "beslut_normalized only known values",
        set(applications["beslut_normalized"].dropna().unique())
        <= {"approved", "rejected", "withdrawn"},
        str(applications["beslut_normalized"].value_counts().to_dict()),
    )
else:
    record("beslut_normalized fully populated", False, "column missing")
    record("beslut_normalized only known values", False, "column missing")

if has_column("yh_poang"):
    record(
        "yh_poang in realistic range (100-1000)",
        applications["yh_poang"].dropna().between(100, 1000).all(),
        f"min={applications['yh_poang'].min()}, max={applications['yh_poang'].max()}",
    )
else:
    record("yh_poang in realistic range (100-1000)", False, "column missing")

if has_column("studietakt_procent"):
    record(
        "studietakt_procent in range (1-200)",
        applications["studietakt_procent"].dropna().between(1, 200).all(),
        f"min={applications['studietakt_procent'].min()}, max={applications['studietakt_procent'].max()}",
    )
else:
    record("studietakt_procent in range (1-200)", False, "column missing")

if has_column("diarienummer"):
    record(
        "diarienummer non-null",
        applications["diarienummer"].isnull().sum() == 0,
        f"{applications['diarienummer'].isnull().sum()} nulls",
    )
else:
    record("diarienummer non-null", False, "column missing")

if has_column("huvudmannatyp"):
    record(
        "No Landsting values remain",
        "Landsting" not in applications["huvudmannatyp"].dropna().unique(),
        f"Unique values: {sorted(applications['huvudmannatyp'].dropna().unique().tolist())}",
    )
else:
    record("No Landsting values remain", False, "column missing")

qc_df = pd.DataFrame(quality_checks).set_index("Check")


def _style_status(value: str) -> str:
    if value == "Pass":
        return "background-color: rgba(46, 204, 113, 0.25); font-weight: 600"
    if value == "Fail":
        return "background-color: rgba(231, 76, 60, 0.25); font-weight: 600"
    return ""


passed = sum(1 for r in quality_checks if r["Status"] == "Pass")
print(f"{passed} / {len(quality_checks)} checks passed\n")
display(qc_df.style.map(_style_status, subset=["Status"]))

# Forward each validation result to the pipeline log so the final report
# reflects quality check outcomes in one place.
for check_result in quality_checks:
    log_event(
        "validate",
        "ok" if check_result["Status"] == "Pass" else "error",
        check_result["Check"],
        check_result["Detail"],
    )

8 / 8 checks passed



,Status,Detail
Check,,
All years loaded,Pass,8 / 8 years present
No completely empty rows,Pass,0 fully-null rows
beslut_normalized fully populated,Pass,0 nulls
beslut_normalized only known values,Pass,"{'rejected': 6391, 'approved': 3591, 'withdrawn': 1}"
yh_poang in realistic range (100-1000),Pass,"min=100, max=999"
studietakt_procent in range (1-200),Pass,"min=50, max=100"
diarienummer non-null,Pass,0 nulls
No Landsting values remain,Pass,"Unique values: ['Kommun', 'Privat', 'Region', 'Statlig']"


---
## 7. Export

The curated dataset is exported to CSV. This file is the single source of truth for the PostgreSQL load script and any downstream consumers.

In [10]:
output_file = OUT_PATH / "curated_applications.csv"
file_size_kb = 0.0

try:
    # utf-8-sig writes a UTF-8 BOM at the start of the file:
    #   - Excel on Windows detects the BOM and opens Swedish characters correctly
    #   - PostgreSQL COPY and pd.read_csv(encoding="utf-8-sig") both strip the BOM on read
    # index=False omits the pandas row index, which has no meaning in the canonical schema.
    applications.to_csv(output_file, index=False, encoding="utf-8-sig")
    file_size_kb = output_file.stat().st_size / 1024
    if file_size_kb < 100:
        log_event(
            "export",
            "warning",
            f"Output file smaller than expected ({file_size_kb:.0f} KB)",
            f"{len(applications):,} rows written - check upstream steps for data loss",
        )
    else:
        log_event(
            "export",
            "ok",
            f"CSV written ({file_size_kb:.0f} KB, {len(applications):,} rows)",
            str(output_file),
        )
except OSError as exc:
    log_event("export", "error", "Failed to write CSV", str(exc))
    raise

print("Export complete")
print(f"  Path     : {output_file}")
print(f"  Rows     : {len(applications):,}")
print(f"  Columns  : {len(applications.columns)}")
print(f"  Size     : {file_size_kb:.0f} KB")
print("  Encoding : utf-8-sig (BOM - compatible with Excel and PostgreSQL COPY)")
print("\nColumn order:")
for i, col in enumerate(applications.columns, 1):
    print(f"  {i:02d}. {col}")

Export complete
  Path     : myh-data-pipeline\data\curated_applications.csv
  Rows     : 9,983
  Columns  : 26
  Size     : 2804 KB
  Encoding : utf-8-sig (BOM - compatible with Excel and PostgreSQL COPY)

Column order:
  01. source_year
  02. source_file
  03. source_sheet
  04. diarienummer
  05. utbildningsnamn
  06. utbildningsomrade
  07. beslut
  08. lan
  09. kommun
  10. yh_poang
  11. studieform
  12. studietakt_procent
  13. utbildningsanordnare
  14. huvudmannatyp
  15. flera_kommuner
  16. antal_kommuner
  17. sokta_omgangar
  18. beviljade_omgangar
  19. typ_av_examen
  20. sun5_inriktning
  21. sun5_inriktning_namn
  22. seqf_niva
  23. smalt_yrkesomrade
  24. beslut_normalized
  25. is_distance
  26. has_multiple_municipalities


---
## 8. Pipeline Report

A full record of every event logged during this run - one row per check, load, or transformation step.
Green = ok, amber = warning, red = error.

This is the single place to look when something is off. The pipeline keeps running past warnings
and logs them here rather than crashing, so this table is always populated even after a partial failure.

In [11]:
if not PIPELINE_LOG:
    print("No events logged.")
else:
    log_df = pd.DataFrame(PIPELINE_LOG)

    ok_count = int((log_df["Level"] == "ok").sum())
    warn_count = int((log_df["Level"] == "warning").sum())
    err_count = int((log_df["Level"] == "error").sum())

    print(
        f"Pipeline complete  |  {ok_count} ok  |  {warn_count} warning(s)  |  {err_count} error(s)"
    )
    if err_count:
        print(
            "  -> Action required: review the errors below before using the exported data."
        )
    elif warn_count:
        print("  -> Warnings present: data may be usable but check the details.")
    else:
        print("  -> All events clean - data is ready for use.")
    print()

    def _style_level(value: str) -> str:
        if value == "ok":
            return "background-color: rgba(46, 204, 113, 0.18)"
        if value == "warning":
            return "background-color: rgba(230, 126, 34, 0.28); font-weight: 600"
        if value == "error":
            return "background-color: rgba(231, 76, 60, 0.28); font-weight: 600"
        return ""

    display(log_df.style.map(_style_level, subset=["Level"]))

Pipeline complete  |  27 ok  |  5 warning(s)  |  0 error(s)
  -> Warnings present: data may be usable but check the details.



,Step,Level,Message,Detail
0,setup,ok,All 8 source files found,
1,harmonize,ok,2018: harmonized,"1105 rows, 14 columns"
2,harmonize,warning,2018: 9 KEEP_COLUMNS absent after rename,"antal_kommuner, beviljade_omgangar, flera_kommuner, seqf_niva, smalt_yrkesomrade, sokta_omgangar, sun5_inriktning, sun5_inriktning_namn, typ_av_examen"
3,harmonize,ok,2019: harmonized,"1237 rows, 16 columns"
4,harmonize,warning,2019: 7 KEEP_COLUMNS absent after rename,"beviljade_omgangar, seqf_niva, smalt_yrkesomrade, sokta_omgangar, sun5_inriktning, sun5_inriktning_namn, typ_av_examen"
5,harmonize,ok,2020: harmonized,"1482 rows, 18 columns"
6,harmonize,warning,2020: 5 KEEP_COLUMNS absent after rename,"seqf_niva, smalt_yrkesomrade, sun5_inriktning, sun5_inriktning_namn, typ_av_examen"
7,harmonize,ok,2021: harmonized,"1238 rows, 19 columns"
8,harmonize,warning,2021: 4 KEEP_COLUMNS absent after rename,"seqf_niva, smalt_yrkesomrade, sun5_inriktning, sun5_inriktning_namn"
9,harmonize,ok,2022: harmonized,"1207 rows, 19 columns"


---
## 9. Reflection

### What went well

The column naming differences across years turned out to be systematic rather than random , once the mapping table was established it handled all variants cleanly. The `Landsting -> Region` normalization and the dual `Beslut` vocabulary (`Ej beviljad` vs `Avslag`) were the most meaningful semantic changes to document.

Using a single `COLUMN_MAP` dictionary and a `KEEP_COLUMNS` list as the contract between raw data and the canonical schema meant the ETL loop itself stayed simple. Adding a new year would require only a new entry in `YEAR_CONFIG` and any new column variants in `COLUMN_MAP`.

### Limitations

- **`typ_av_examen` is null for 2018 and 2020.** The column existed in 2019 and 2021+ but not in the 2018 file or the 2020 Tabell 3 variant. This is a source limitation, not a pipeline error.
- **Duplicate `diarienummer` values exist across years** , the same program can apply in multiple rounds. This is expected behaviour. Filtering by `source_year` gives clean per-round analysis.
- **2018–2019 are missing `sokta_omgangar` and `beviljade_omgangar`.** The applied and approved rounds count was introduced in the 2020 file format.

### Design decisions

- SUN5, SeQF, and smalt yrkesområde columns are retained as nulls for pre-2023 rows rather than being dropped. A null is informative; dropping a column would hide the fact that the data exists at all for newer years.
- 2016–2017 were considered and excluded. 2017 is borderline usable, but 2016 is missing Studieform and Huvudmannatyp entirely , including it would introduce a two-year gap in those fields that could silently bias any study-form analysis.
- Values are kept in Swedish in the base columns (`beslut`, `studieform`, `utbildningsomrade`). Only `beslut_normalized` is translated to English , this preserves the original data while providing a clean label for API consumers.